# FinIA-Flex — Paso 3: Índice Vectorial (RAG)

Caso Práctico Unidad 1, materia Generative IA — Maestría en Ciencia de Datos y Analítica
Visual, IEP.

Aquí construyo el índice vectorial para que el sistema pueda buscar automáticamente en las
políticas internas antes de generar un reporte, en vez de que yo tenga que pegarle el
contexto a mano cada vez.

Voy a: instalar dependencias, cargar los documentos (Markdown y PDF), fragmentarlos, generar
embeddings, meterlos a ChromaDB, y probar que la búsqueda funcione.

Nota rápida sobre el formato: mis documentos de política están en Markdown, simulando el
resultado de convertir un PDF/Word real a texto plano (eso normalmente se haría con
`PyPDFLoader` o `Docx2txtLoader`). Lo hice así para no perder tiempo en la parte de extracción
de texto y enfocarme en RAG, prompting y fine-tuning, que es lo que pide el caso. En la
Sección 2.2 sí cargo un PDF real para probar que el pipeline también lo soporta.

Usé ChromaDB en vez de FAISS porque es más simple de usar en un notebook y con solo 5
documentos no necesito las optimizaciones de FAISS.


## 1. Instalación de dependencias

Si Colab reinicia el entorno de ejecución tras instalar `chromadb`, basta con ejecutar nuevamente las celdas desde este punto.

In [ ]:
!pip install -q langchain langchain-community langchain-chroma langchain-text-splitters chromadb sentence-transformers pypdf tiktoken
print("Instalación completa.")

## 2. Carga de los documentos fuente

Los documentos deben colocarse en una carpeta de Google Drive con la siguiente estructura:

```
FinIA-Flex/
└── rag_docs/
    ├── 01_Politica_Gasto_Aprobaciones.md
    ├── 01_Politica_Gasto_Aprobaciones.pdf   <- versión PDF del mismo documento
    ├── 02_Principios_Costeo_Variaciones.md
    ├── 03_Formato_Reporte_Ejecutivo.md
    └── 04_Buenas_Practicas_Optimizacion_Costos.md
```

Como alternativa, los archivos pueden cargarse directamente en la sesión de Colab (panel
Archivos), sin necesidad de montar Google Drive; en ese caso no persisten al cerrar la sesión.


In [ ]:
# Montaje de Google Drive
from google.colab import drive
drive.mount('/content/drive')

# Ruta de la carpeta de documentos. Modificar si la estructura de carpetas es distinta.
RAG_DOCS_PATH = "/content/drive/MyDrive/FinIA-Flex/rag_docs"

# Alternativa sin Google Drive (carga manual en el panel Archivos de Colab):
# RAG_DOCS_PATH = "/content/rag_docs"

import os
archivos = os.listdir(RAG_DOCS_PATH)
print(f"Archivos encontrados en {RAG_DOCS_PATH}:")
for a in archivos:
    print(" -", a)

### 2.1 Carga de documentos Markdown

In [ ]:
from langchain_community.document_loaders import DirectoryLoader, TextLoader

loader_md = DirectoryLoader(
    RAG_DOCS_PATH,
    glob="*.md",
    loader_cls=TextLoader,
    loader_kwargs={"encoding": "utf-8"},
)

documentos_md = loader_md.load()
print(f"Documentos Markdown cargados: {len(documentos_md)}")
for d in documentos_md:
    print(" -", d.metadata["source"], f"({len(d.page_content)} caracteres)")

### 2.2 Cargando también un PDF (para probar que sí se puede)

Quería confirmar que el pipeline no solo funciona con Markdown, así que cargo la versión PDF
de la Política de Gasto con `PyPDFLoader` — así se vería en un caso real donde los documentos
llegan en PDF o Word.


In [ ]:
from langchain_community.document_loaders import PyPDFLoader

ruta_pdf = os.path.join(RAG_DOCS_PATH, "01_Politica_Gasto_Aprobaciones.pdf")

loader_pdf = PyPDFLoader(ruta_pdf)
documentos_pdf = loader_pdf.load()

print(f"Páginas cargadas desde PDF: {len(documentos_pdf)}")
print("\n--- Extracto de la primera página ---")
print(documentos_pdf[0].page_content[:600])

### 2.3 ¿Cuál uso al final?

Para no duplicar contenido, solo indexo la versión Markdown; el PDF de arriba lo dejo nada
más como prueba de que el loader funciona.


In [ ]:
# Documentos que se indexarán: los 4 documentos en Markdown.
# (documentos_pdf queda disponible como evidencia de soporte multi-formato, sin duplicar contenido)
documentos = documentos_md
print(f"Total de documentos a indexar: {len(documentos)}")

## 3. Fragmentación (chunking)

Uso `RecursiveCharacterTextSplitter` respetando los encabezados Markdown, para que un
fragmento no se corte a la mitad de una regla — por ejemplo, que el umbral de $50,000 no
quede separado de la categoría a la que aplica.

Elegí `chunk_size=500` y `chunk_overlap=80` porque mis documentos tienen secciones cortas; un
tamaño mayor mezclaría reglas de categorías distintas en un mismo fragmento.

Aquí me topé con mi primer error real: `from langchain.text_splitter import ...` ya no
funciona en versiones recientes de LangChain — ahora está en un paquete aparte,
`langchain-text-splitters`. Lo corregí cambiando el import.


In [ ]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=80,
    separators=["\n## ", "\n### ", "\n\n", "\n", ". ", " "],
)

fragmentos = splitter.split_documents(documentos)
print(f"Total de fragmentos generados: {len(fragmentos)}")
print("\n--- Ejemplo de fragmento ---")
print(fragmentos[0].page_content)
print("\nFuente:", fragmentos[0].metadata["source"])

## 4. Embeddings y base vectorial

Uso `all-MiniLM-L6-v2` de sentence-transformers — es local y gratis, y no tiene que ver con
el LLM que voy a usar después para generar texto (eso es aparte, en el Paso 5).


In [ ]:
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_chroma import Chroma

embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")

CHROMA_PATH = "/content/finia_flex_chroma_db"

vectorstore = Chroma.from_documents(
    documents=fragmentos,
    embedding=embeddings,
    persist_directory=CHROMA_PATH,
)

print(f"Base vectorial creada con {vectorstore._collection.count()} fragmentos indexados.")

## 5. Prueba de búsqueda

Pruebo con preguntas relacionadas a los 3 escenarios de mi dataset (Paso 1), para confirmar
que trae el fragmento correcto y no algo genérico.


In [ ]:
preguntas_prueba = [
    "¿Qué umbral de gasto en mantenimiento requiere aprobación gerencial?",
    "¿Qué se debe hacer si un centro de costo tiene sobrecosto 3 meses seguidos?",
    "¿Un ahorro grande siempre es una buena noticia?",
    "¿Cómo se debe estructurar un reporte ejecutivo de variación de presupuesto?",
]

for pregunta in preguntas_prueba:
    print("=" * 80)
    print("PREGUNTA:", pregunta)
    resultados = vectorstore.similarity_search(pregunta, k=2)
    for i, r in enumerate(resultados, start=1):
        print(f"\n--- Resultado {i} (fuente: {r.metadata['source'].split('/')[-1]}) ---")
        print(r.page_content[:400])
    print()

## 6. Guardar el índice

Copio la carpeta a Drive para que no se pierda al cerrar la sesión — la voy a necesitar en el
Paso 5.


In [ ]:
import shutil

DRIVE_BACKUP_PATH = "/content/drive/MyDrive/FinIA-Flex/finia_flex_chroma_db"
shutil.copytree(CHROMA_PATH, DRIVE_BACKUP_PATH, dirs_exist_ok=True)
print("Base vectorial respaldada en:", DRIVE_BACKUP_PATH)

---
## Resumen

Construí el índice RAG a partir de mis 4 documentos de política, con ChromaDB y embeddings
gratuitos. En el camino me encontré con el error de import de `langchain-text-splitters` (ya
corregido arriba). Las pruebas de la Sección 5 muestran que sí trae el fragmento correcto para
cada pregunta.

Siguiente: Paso 4, diseñar el prompt que va a usar este contexto para redactar el reporte.
